In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set clean aesthetic styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 11, "figure.autolayout": True})

# 1. Load data from the sweep directory
sweep_dir = Path("../benchmarks/team-sampler-sweep")
data = []

for folder in sweep_dir.glob("*"):
    summary_path = folder / "summary.json"
    if not summary_path.exists():
        # Fallback search inside subdirectories
        summary_files = list(folder.rglob("summary.json"))
        if summary_files:
            summary_path = summary_files[0]
        else:
            continue

    with open(summary_path, "r") as f:
        content = json.load(f)

    # Get the backend dictionary (e.g., team_gumbel_topk or team_sampler)
    backends = content.get("backends", {})
    backend_key = list(backends.keys())[0]
    agg = backends[backend_key]["aggregate"]

    # Extract score, 95% CI bounds, and KV access percentage
    score = agg["score"]
    ci_lower = agg["score_ci_lower"]
    ci_upper = agg["score_ci_upper"]
    kv_access_pct = agg["decode_gqa_total_access_pct"]

    # Extract algorithm hyperparameters
    algo = agg
    P = algo["parent_size"]
    R = algo["representatives_per_parent"]
    S = algo["samples_per_head"]

    data.append(
        {
            "folder": folder.name,
            "P": P,
            "R": R,
            "S": S,
            "score": score,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "err_minus": score - ci_lower,
            "err_plus": ci_upper - score,
            "kv_access_pct": kv_access_pct,
        }
    )

df = pd.DataFrame(data)
print("Loaded Run Summary Data:\n", df[["folder", "P", "R", "S", "score", "kv_access_pct"]])

# ==============================================================================
# PLOT SET 1: Sweep Nominal Samples S (P=16, R=4)
# ==============================================================================
df_s = df[(df["P"] == 16) & (df["R"] == 4)].sort_values("S")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.5, 1]})

# Line plot with 95% Bootstrap Error Bars vs. KV Access %
ax1.errorbar(
    df_s["kv_access_pct"],
    df_s["score"],
    yerr=[df_s["err_minus"], df_s["err_plus"]],
    fmt="-o",
    color="#1f77b4",
    linewidth=2,
    markersize=8,
    capsize=5,
    capthick=1.5,
    label="P=16, R=4",
)

for _, row in df_s.iterrows():
    ax1.annotate(
        f"S={row['S']}",
        (row["kv_access_pct"], row["score"]),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontweight="bold",
    )

ax1.set_xlabel("Memory Traffic / KV Access (%)")
ax1.set_ylabel("Mean Task Accuracy (%)")
ax1.set_title("Task Accuracy vs. Memory Access (Nominal Samples S Sweep)")
ax1.grid(True, linestyle="--", alpha=0.6)

# Heatmap for S Sweep
pivot_s = df_s.pivot(index="S", columns="P", values="score")
sns.heatmap(
    pivot_s,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    cbar_kws={"label": "Mean Task Accuracy (%)"},
    ax=ax2,
    linewidths=1,
)
ax2.set_title("Accuracy Heatmap (S Sweep, P=16, R=4)")
ax2.set_xlabel("Parent Size (P)")
ax2.set_ylabel("Nominal Samples (S)")

plt.tight_layout()
plt.savefig("sweep_samples_S_with_ci.png", dpi=300)
plt.close()

# ==============================================================================
# PLOT SET 2: Sweep Parent Size P across R=4 and R=8 (S=128)
# ==============================================================================
df_p = df[df["S"] == 128].sort_values(["R", "P"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1.5, 1]})

colors = {4: "#1f77b4", 8: "#ff7f0e"}
markers = {4: "-o", 8: "-s"}

for r_val in [4, 8]:
    sub_df = df_p[df_p["R"] == r_val]
    if sub_df.empty:
        continue

    ax1.errorbar(
        sub_df["kv_access_pct"],
        sub_df["score"],
        yerr=[sub_df["err_minus"], sub_df["err_plus"]],
        fmt=markers[r_val],
        color=colors[r_val],
        linewidth=2,
        markersize=8,
        capsize=5,
        capthick=1.5,
        label=f"R = {r_val}",
    )

    for _, row in sub_df.iterrows():
        ax1.annotate(
            f"P={row['P']}",
            (row["kv_access_pct"], row["score"]),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontweight="bold",
        )

ax1.set_xlabel("Memory Traffic / KV Access (%)")
ax1.set_ylabel("Mean Task Accuracy (%)")
ax1.set_title("Task Accuracy vs. Memory Access (Parent Size P & Reps R Sweep)")
ax1.legend(title="Representatives (R)")
ax1.grid(True, linestyle="--", alpha=0.6)

# Heatmap for (P, R) Sweep
pivot_pr = df_p.pivot(index="R", columns="P", values="score")
sns.heatmap(
    pivot_pr,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    cbar_kws={"label": "Mean Task Accuracy (%)"},
    ax=ax2,
    linewidths=1,
)
ax2.set_title("Accuracy Heatmap (P vs. R at S=128)")
ax2.set_xlabel("Parent Size (P)")
ax2.set_ylabel("Representatives per Parent (R)")

plt.tight_layout()
plt.savefig("sweep_parent_P_reps_R_with_ci.png", dpi=300)
plt.close()

print("Plots successfully saved: 'sweep_samples_S_with_ci.png' & 'sweep_parent_P_reps_R_with_ci.png'")

Loaded Run Summary Data:
         folder   P  R    S  score  kv_access_pct
0  P16-R4-S128  16  4  128  96.00      25.845996
1   P16-R4-S32  16  4   32  87.75      16.595321
2   P16-R4-S64  16  4   64  93.50      19.910875
3  P32-R4-S128  32  4  128  90.88      20.006761
4  P32-R8-S128  32  8  128  95.62      26.206795
5  P64-R4-S128  64  4  128  55.00      17.324239
6  P64-R8-S128  64  8  128  88.62      21.667902
Plots successfully saved: 'sweep_samples_S_with_ci.png' & 'sweep_parent_P_reps_R_with_ci.png'


In [3]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set clean aesthetic styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 10, "figure.autolayout": True})

# 1. Load data from the sweep directory
sweep_dir = Path("../benchmarks/team-sampler-sweep")
data = []

for folder in sweep_dir.glob("*"):
    summary_path = folder / "summary.json"
    if not summary_path.exists():
        summary_files = list(folder.rglob("summary.json"))
        if summary_files:
            summary_path = summary_files[0]
        else:
            continue

    with open(summary_path, "r") as f:
        content = json.load(f)

    backends = content.get("backends", {})
    backend_key = list(backends.keys())[0]
    backend_data = backends[backend_key]
    agg = backend_data["aggregate"]

    # Overall Mean
    score = agg["score"]
    ci_lower = agg["score_ci_lower"]
    ci_upper = agg["score_ci_upper"]
    kv_access_pct = agg["decode_gqa_total_access_pct"]

    # Hyperparameters
    P = agg["parent_size"]
    R = agg["representatives_per_parent"]
    S = agg["samples_per_head"]

    row_dict = {
        "folder": folder.name,
        "P": P,
        "R": R,
        "S": S,
        "Overall Mean": score,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "err_minus": score - ci_lower,
        "err_plus": ci_upper - score,
        "kv_access_pct": kv_access_pct,
    }

    # Extract individual task breakdown scores
    tasks = backend_data.get("tasks", {})
    for task_name, task_info in tasks.items():
        row_dict[task_name] = task_info["score"]

    data.append(row_dict)

df = pd.DataFrame(data)

# Detect task names automatically (excluding overall aggregate metrics)
non_task_cols = {
    "folder", "P", "R", "S", "Overall Mean", "ci_lower", "ci_upper",
    "err_minus", "err_plus", "kv_access_pct"
}
task_cols = [col for col in df.columns if col not in non_task_cols]
score_cols = ["Overall Mean"] + task_cols

print("Loaded Run Summary Data:\n", df[["folder", "P", "R", "S"] + score_cols + ["kv_access_pct"]])

# ==============================================================================
# PLOT SET 1: Sweep Nominal Samples S (P=16, R=4)
# ==============================================================================
df_s = df[(df["P"] == 16) & (df["R"] == 4)].sort_values("S")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1.2, 1]})

# Line plot with 95% Bootstrap Error Bars
ax1.errorbar(
    df_s["kv_access_pct"],
    df_s["Overall Mean"],
    yerr=[df_s["err_minus"], df_s["err_plus"]],
    fmt="-o",
    color="#1f77b4",
    linewidth=2,
    markersize=8,
    capsize=5,
    capthick=1.5,
    label="P=16, R=4",
)

for _, row in df_s.iterrows():
    ax1.annotate(
        f"S={row['S']}",
        (row["kv_access_pct"], row["Overall Mean"]),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontweight="bold",
    )

ax1.set_xlabel("Memory Traffic / KV Access (%)")
ax1.set_ylabel("Mean Task Accuracy (%)")
ax1.set_title("Task Accuracy vs. Memory Access (Nominal Samples S Sweep)")
ax1.grid(True, linestyle="--", alpha=0.6)

# Heatmap with Task Breakdown for S Sweep
df_s_heatmap = df_s.set_index("S")[score_cols]
sns.heatmap(
    df_s_heatmap,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    cbar_kws={"label": "Accuracy (%)"},
    ax=ax2,
    linewidths=1,
)
ax2.set_title("Accuracy Breakdown Heatmap (S Sweep, P=16, R=4)")
ax2.set_ylabel("Nominal Samples (S)")
ax2.set_xlabel("Tasks & Aggregate")

plt.tight_layout()
plt.savefig("sweep_samples_S_with_breakdown.png", dpi=300)
plt.close()

# ==============================================================================
# PLOT SET 2: Sweep Parent Size P across R=4 and R=8 (S=128)
# ==============================================================================
df_p = df[df["S"] == 128].sort_values(["R", "P"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.2, 1]})

colors = {4: "#1f77b4", 8: "#ff7f0e"}
markers = {4: "-o", 8: "-s"}

for r_val in [4, 8]:
    sub_df = df_p[df_p["R"] == r_val]
    if sub_df.empty:
        continue

    ax1.errorbar(
        sub_df["kv_access_pct"],
        sub_df["Overall Mean"],
        yerr=[sub_df["err_minus"], sub_df["err_plus"]],
        fmt=markers[r_val],
        color=colors[r_val],
        linewidth=2,
        markersize=8,
        capsize=5,
        capthick=1.5,
        label=f"R = {r_val}",
    )

    for _, row in sub_df.iterrows():
        ax1.annotate(
            f"P={row['P']}",
            (row["kv_access_pct"], row["Overall Mean"]),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontweight="bold",
        )

ax1.set_xlabel("Memory Traffic / KV Access (%)")
ax1.set_ylabel("Mean Task Accuracy (%)")
ax1.set_title("Task Accuracy vs. Memory Access (Parent Size P & Reps R Sweep)")
ax1.legend(title="Representatives (R)")
ax1.grid(True, linestyle="--", alpha=0.6)

# Heatmap with Task Breakdown for (P, R) Sweep
df_p["Configuration (P, R)"] = df_p.apply(lambda r: f"P={r['P']}, R={r['R']}", axis=1)
df_p_heatmap = df_p.set_index("Configuration (P, R)")[score_cols]

sns.heatmap(
    df_p_heatmap,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    cbar_kws={"label": "Accuracy (%)"},
    ax=ax2,
    linewidths=1,
)
ax2.set_title("Accuracy Breakdown Heatmap (P & R Sweep at S=128)")
ax2.set_ylabel("Configuration")
ax2.set_xlabel("Tasks & Aggregate")

plt.tight_layout()
plt.savefig("sweep_parent_P_reps_R_with_breakdown.png", dpi=300)
plt.close()

print("Plots successfully saved: 'sweep_samples_S_with_breakdown.png' & 'sweep_parent_P_reps_R_with_breakdown.png'")

Loaded Run Summary Data:
         folder   P  R    S  Overall Mean  niah_multiquery  niah_multivalue  \
0  P16-R4-S128  16  4  128         96.00            99.25            92.75   
1   P16-R4-S32  16  4   32         87.75            91.50            84.00   
2   P16-R4-S64  16  4   64         93.50            96.25            90.75   
3  P32-R4-S128  32  4  128         90.88            95.50            86.25   
4  P32-R8-S128  32  8  128         95.62            99.00            92.25   
5  P64-R4-S128  64  4  128         55.00            60.00            50.00   
6  P64-R8-S128  64  8  128         88.62            93.00            84.25   

   kv_access_pct  
0      25.845996  
1      16.595321  
2      19.910875  
3      20.006761  
4      26.206795  
5      17.324239  
6      21.667902  
Plots successfully saved: 'sweep_samples_S_with_breakdown.png' & 'sweep_parent_P_reps_R_with_breakdown.png'
